### Lab 1: Knapsack Problem

Copyright **`(c)`** 2025 Giovanni Squillero `<giovanni.squillero@polito.it>`,
    [`https://github.com/squillero/computational-intelligence`](https://github.com/squillero/computational-intelligence),
    Free under certain conditions — see the [`license`](https://github.com/squillero/computational-intelligence/blob/master/LICENSE.md) for details.  

In [1]:
import numpy as np

In [29]:
NUM_KNAPSACKS = 2
NUM_ITEMS = 10
NUM_DIMENSIONS = 2

In [ ]:
VALUES = np.random.randint(0, 100, size=NUM_ITEMS)
WEIGHTS = np.random.randint(0, 100, size=(NUM_ITEMS, NUM_DIMENSIONS))
CONSTRAINTS = np.random.randint(
    0, 100 * NUM_ITEMS // NUM_KNAPSACKS, size=(NUM_KNAPSACKS, NUM_DIMENSIONS)
)

In [46]:
CONSTRAINTS

array([614, 496])

In [32]:
# A random solution
solution = np.array(
    [np.random.random(NUM_ITEMS) < 0.5 for _ in range(NUM_KNAPSACKS)], dtype=np.bool
)

In [33]:
solution

array([[False, False, False,  True,  True, False, False, False, False,
        False],
       [False,  True,  True, False,  True,  True,  True,  True,  True,
        False]])

In [34]:

# Check that the same object does not appear in multiple knapsacks
np.all(solution.sum(axis=0) <= 1)

np.False_

In [35]:

# Check if the solution is valid
all_knapsacks = np.any(solution, axis=0)
np.all(WEIGHTS[all_knapsacks].sum(axis=0) < CONSTRAINTS)
print("Weights:", WEIGHTS[all_knapsacks].sum(axis=0))
print("Constraints:", CONSTRAINTS)

Weights: [405 419]
Constraints: [194  15]


## TEST PROBLEMS

In [47]:

# Problem 1:
rng = np.random.default_rng(seed=42)
NUM_KNAPSACKS = 3
NUM_ITEMS = 20
NUM_DIMENSIONS = 2
VALUES = rng.integers(0, 100, size=NUM_ITEMS)
WEIGHTS = rng.integers(0, 100, size=(NUM_ITEMS, NUM_DIMENSIONS))
CONSTRAINTS = rng.integers(
    0, 100 * NUM_ITEMS // NUM_KNAPSACKS, size=(NUM_KNAPSACKS, NUM_DIMENSIONS)
)

In [ ]:
# Problem 2:
rng = np.random.default_rng(seed=42)
NUM_KNAPSACKS = 10
NUM_ITEMS = 100
NUM_DIMENSIONS = 10
VALUES = rng.integers(0, 1000, size=NUM_ITEMS)
WEIGHTS = rng.integers(0, 1000, size=(NUM_ITEMS, NUM_DIMENSIONS))
CONSTRAINTS = rng.integers(
    1000 * 2, 1000 * NUM_ITEMS // NUM_KNAPSACKS, size=(NUM_KNAPSACKS, NUM_DIMENSIONS)
)

In [ ]:

# Problem 3:
rng = np.random.default_rng(seed=42)
NUM_KNAPSACKS = 100
NUM_ITEMS = 5000
NUM_DIMENSIONS = 100
VALUES = rng.integers(0, 1000, size=NUM_ITEMS)
WEIGHTS = rng.integers(0, 1000, size=(NUM_ITEMS, NUM_DIMENSIONS))
CONSTRAINTS = rng.integers(
    1000 * 10, 1000 * 2 * NUM_ITEMS // NUM_KNAPSACKS, size=(NUM_KNAPSACKS, NUM_DIMENSIONS)

## Solution

In [48]:
print(VALUES)

print(WEIGHTS)

print(CONSTRAINTS)

[ 8 77 65 43 43 85  8 69 20  9 52 97 73 76 71 78 51 12 83 45]
[[50 37]
 [18 92]
 [78 64]
 [40 82]
 [54 44]
 [45 22]
 [ 9 55]
 [88  6]
 [85 82]
 [27 63]
 [16 75]
 [70 35]
 [ 6 97]
 [44 89]
 [67 77]
 [75 19]
 [36 46]
 [49  4]
 [54 15]
 [74 68]]
[[614 496]
 [244 644]
 [273 216]]


In [59]:
# Generate a random initial solution
initial_solution = np.array(
    [rng.random(NUM_ITEMS) < 0.5 for _ in range(NUM_KNAPSACKS)], dtype=np.bool
)
initial_solution

array([[False, False, False, False,  True, False, False,  True,  True,
        False,  True, False,  True,  True, False,  True,  True, False,
        False,  True],
       [ True, False, False, False,  True,  True,  True,  True, False,
         True, False, False, False,  True, False, False, False,  True,
         True, False],
       [ True,  True,  True, False, False,  True, False,  True, False,
        False,  True,  True,  True, False, False, False, False,  True,
        False,  True]])

In [60]:
def check_solution(solution, debug = 0) -> bool:
    # Check that the same object does not appear in multiple knapsacks
    if not np.all(solution.sum(axis=0) <= 1):
        if debug:
            print("Invalid solution: same object appears in multiple knapsacks")
        return False

    total_weights = np.zeros((NUM_KNAPSACKS, NUM_DIMENSIONS), dtype=np.int32)
    # Check constraints for each knapsack individually
    for k in range(NUM_KNAPSACKS):
        items_in_knapsack = np.where(solution[k])[0]
        total_weights[k] = WEIGHTS[items_in_knapsack].sum(axis=0)

    if not np.all(total_weights < CONSTRAINTS):
        if debug:
            print("Invalid solution: knapsack constraints violated")
            print("Weights:", total_weights)
            print("Constraints:", CONSTRAINTS)
        return False

    return True

In [66]:
def tweak_solution(solution):
    new_solution = solution.copy()
    # Randomly choose to add or remove an item
    if np.random.rand() < 0.5:
        # Add an item to a random knapsack
        knapsack = np.random.randint(NUM_KNAPSACKS)
        item = np.random.randint(NUM_ITEMS)
        # Ensure the item is not already in another knapsack
        if not np.any(new_solution[:, item]):
            new_solution[knapsack, item] = True
    else:
        # Remove an item from a random knapsack
        knapsack = np.random.randint(NUM_KNAPSACKS)
        items_in_knapsack = np.where(new_solution[knapsack])[0]
        if len(items_in_knapsack) > 0:
            item = np.random.choice(items_in_knapsack)
            new_solution[knapsack, item] = False
    return new_solution

In [61]:
# Check that the same object does not appear in multiple knapsacks
if not np.all(initial_solution.sum(axis=0) <= 1):
    indices = np.where(initial_solution.sum(axis=0) > 1)[0]
    for i in indices:
        knapsacks = np.where(initial_solution[:, i])[0]
        to_remove = rng.choice(knapsacks, size=len(knapsacks) - 1, replace=False)
        initial_solution[to_remove, i] = False
np.all(initial_solution.sum(axis=0) <= 1)

total_weights = np.zeros((NUM_KNAPSACKS, NUM_DIMENSIONS), dtype=np.int32)
# Check constraints for each knapsack individually
for k in range(NUM_KNAPSACKS):
    items_in_knapsack = initial_solution[k]
    total_weights[k] = WEIGHTS[items_in_knapsack].sum(axis=0)


if not np.all(total_weights < CONSTRAINTS):
    print(f"solution  is not valid")
    print("Weights:", total_weights)
    print("Constraints:", CONSTRAINTS)
else:
    print(f"Knapsack is valid")

# we want to create a valid initial solution so we will remove items until the solution is valid
while not np.all(total_weights < CONSTRAINTS):
    # Find a knapsack that is not valid
    invalid_knapsacks = np.where(np.any(total_weights >= CONSTRAINTS, axis=1))[0]
    k = rng.choice(invalid_knapsacks)
    # Find items in the knapsack
    items_in_knapsack = np.where(initial_solution[k])[0]
    if len(items_in_knapsack) == 0:
        continue
    # Remove a random item from the knapsack
    item_to_remove = rng.choice(items_in_knapsack)
    initial_solution[k, item_to_remove] = False
    # Update total weights
    total_weights[k] = WEIGHTS[initial_solution[k]].sum(axis=0)


total_weights = np.zeros((NUM_KNAPSACKS, NUM_DIMENSIONS), dtype=np.int32)
# Check constraints for each knapsack individually
for k in range(NUM_KNAPSACKS):
    items_in_knapsack = initial_solution[k]
    total_weights[k] = WEIGHTS[items_in_knapsack].sum(axis=0)


if not np.all(total_weights < CONSTRAINTS):
    print(f"solution  is not valid")
    print("Weights:", total_weights)
    print("Constraints:", CONSTRAINTS)
else:
    print(f"Knapsack is valid")
    print("Weights:", total_weights)
    print("Constraints:", CONSTRAINTS)


solution  is not valid
Weights: [[270 215]
 [287 307]
 [321 391]]
Constraints: [[614 496]
 [244 644]
 [273 216]]
Knapsack is valid
Weights: [[270 215]
 [228 215]
 [221 155]]
Constraints: [[614 496]
 [244 644]
 [273 216]]


In [71]:
print(check_solution(initial_solution, debug=1))

initial_solution_values = np.sum(np.sum(initial_solution * VALUES, axis=1))

print("Initial solution values:", initial_solution_values)

# We can now use initial_solution as a valid starting point for our optimization algorithm
# We will only climb and accept only solution that are valid and better than the current solution
MAX_ITERATIONS = 100000
current_solution = initial_solution
current_solution_values = initial_solution_values

max_values = np.sum(VALUES)
print("Max values:", max_values)

for _ in range(MAX_ITERATIONS or current_solution_values == max_values):
    new_solution = tweak_solution(current_solution)
    if check_solution(new_solution):
        new_solution_values = np.sum(np.sum(new_solution * VALUES, axis=1))
        if new_solution_values > current_solution_values:
            current_solution = new_solution
            current_solution_values = new_solution_values
            print("New best solution values:", current_solution_values)

print("Final solution values:", current_solution_values)

True
Initial solution values: 745
Max values: 1065
New best solution values: 753
New best solution values: 826
New best solution values: 878
New best solution values: 886
New best solution values: 951
Final solution values: 951
